# Forecast-driven reservoir operation in the Lake Tana highlands

**Question.** Ethiopia's reservoirs are its energy storage, and a rainfall forecast is a decision about when to release water.
Does a monthly rainfall forecast let an operator spill less and generate more, and when the rains fail, how much of the gap can solar and wind fill?

**Design: predict, decide, test.**
1. *Predict* monthly rainfall at Lake Tana with a seasonal-naive baseline, climatology, SARIMA and gradient boosting, on years the models never saw.
2. *Decide* with a toy reservoir whose inflow follows lagged rainfall, comparing release rules that use, or ignore, the forecast.
3. *Test* the rules on the held-out years, counting months of shortfall and spilled water.
4. *Back up* with solar and wind from the same NASA POWER cell and measure how much of the hydropower shortfall they cover.

Everything here is a prototype on one grid cell. Reservoir size, demand and plant are illustrative assumptions (see `tana/config.py`).

In [ ]:
import os, io, sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / 'tana').exists():
    ROOT = ROOT.parent          # notebook opened from notebooks/
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import numpy as np, pandas as pd
from IPython.display import Image, display, Markdown
from dataclasses import replace

from tana.config import Config
from tana import plotting, reservoir
from tana.pipeline import load_data, analyse

SYNTHETIC = os.environ.get("TANA_SYNTHETIC") == "1"   # offline smoke test only; leave unset for real results
cfg = Config()
if SYNTHETIC:
    display(Markdown("**SYNTHETIC TEST DATA: nothing below is a real result.**"))

def show(fig):
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=150, bbox_inches="tight"); buf.seek(0)
    display(Image(buf.read()))
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Data: NASA POWER monthly rainfall, solar irradiance and wind

In [ ]:
df = load_data(cfg, synthetic=SYNTHETIC)
print(f"{cfg.lat}N, {cfg.lon}E | {df.index[0]:%Y-%m} to {df.index[-1]:%Y-%m} | {len(df)} months")
df[["rain_mm", "solar_kwh_m2_day", "wind_ms_50m"]].describe().T

Running every step once (forecasts are refitted yearly, so this takes about a minute).

In [ ]:
R = analyse(cfg, df)
show(plotting.fig_climate(R['df'], R['seasons'], cfg.test_start))

Rainfall is sharply seasonal: the June to September rains fill the reservoir and the months either side are dry. Compare how much less the sun and wind vary across the year. The table in section 5 shows whether dry-season sun and wind are meaningfully higher than rainy-season values at this location, which is what decides how good a seasonal backup they are.

## 2. Predict: rainfall forecasts on held-out years

All models forecast 1, 2 and 3 months ahead using only information available at the time. SARIMA and boosting are refitted once a year on data strictly before that year. A unit test (`tests/test_core.py`) checks that changing the test-period data does not change earlier forecasts.

In [ ]:
sc = R['scores']
display(sc.pivot(index='model', columns='horizon', values='rmse_mm').rename_axis(None).round(1))
display(sc[sc.horizon == 1].set_index('model')[['rmse_mm', 'mae_mm', 'rmse_wet_mm', 'skill_vs_climatology']])
show(plotting.fig_forecast(R['rain'], R['fcs'], sc, cfg.test_start, cfg.horizon))

**How to read it.** Skill is measured against climatology (the long-run average for that calendar month). A strongly seasonal series makes climatology hard to beat, so a small or negative skill is an honest result, not a bug. Report whichever it is.

## 3. Decide: a toy reservoir and five release rules

Inflow follows lagged rainfall (35% / 40% / 25% over the current and next two months), scaled to an assumed mean annual inflow. Each month the operator must release a firm demand. The rules differ only in what they know:

| Rule | What the operator uses |
|---|---|
| No forecast | Releases the firm demand, spills only when full |
| Climatology | Looks 3 months ahead using the average rainfall for each month |
| SARIMA / Boosting | Looks 3 months ahead using the model forecast |
| Perfect foresight | Looks 3 months ahead using the true future rainfall (an upper bound) |

Look-ahead rules release extra water through the turbines when the reservoir is projected to overflow, up to the turbine limit. They never cut supply below the firm demand.

In [ ]:
p = R['params']
print(f"capacity {p.cap:,.0f} Mm3 | dead storage {p.dead:,.0f} | firm demand {p.demand:,.0f}/month | max release {p.rmax:,.0f}/month")
show(plotting.fig_rules(R['sims'], p, R['test_index']))

## 4. Test: months of shortfall and spilled water

In [ ]:
display(R['table'][['shortfall_months', 'unmet_mm3', 'spill_mm3', 'spill_months', 'generation_gwh']].round(1))
show(plotting.fig_scorecard(R['table']))

Perfect foresight shows how much any forecast could possibly help. The gap between the forecast rules and that bound is the room left for better forecasts. If shortfall months do not change across rules, that is expected: with a fixed firm demand, the water balance over a dry season is set by what is in storage, and the look-ahead rule mainly saves water that would have been spilled.

### How much does reservoir size change the answer?

In [ ]:
show(plotting.fig_sensitivity(R['sens'], cfg.mean_annual_inflow_mm3 / 12))

## 5. Solar and wind as the backup

From the same cell: how do sun and wind behave in the dry season and in a weak rainy season (June to September rainfall in the lowest quarter of years)?

In [ ]:
display(R['cond'].round(2))
print('Weak rainy seasons:', [int(y) for y in R['seasons'].index[R['seasons'].weak]])
show(plotting.fig_backup(R['cond'], R['seasons'], R['per_mw']))

### Gap cover

The hydropower shortfall comes from running the no-forecast reservoir over the whole 1990 to 2025 record (first two years ignored while storage settles). Solar energy uses peak-sun-hours and a performance ratio. Wind uses a Weibull wind distribution and a power curve. These are screening-level estimates, not a plant design.

In [ ]:
sf = R['shortfall_gwh']
print(f"shortfall months: {(sf > 0).sum()} | total shortfall energy: {sf.sum():,.0f} GWh")
cv = R['cover']
display(cv[cv.mw.isin([25, 50, 100, 150, 200])].set_index('mw').mul(100).round(0).rename(columns=lambda c: f"{c} (% covered)"))
if len(R['req']):
    print(f"Worst month needs ~{R['req'].mw_needed_mix.max():,.0f} MW of a 50/50 solar+wind portfolio to cover it fully.")
show(plotting.fig_gap(sf, R['per_mw'], cv, R['req']))
print('Complementarity (deseasonalised correlations):', {k: round(v, 2) for k, v in R['compl'].items()})

## 6. Battery dispatch: does a day-ahead solar forecast improve the overnight top-up?

A battery beside solar PV is reserved for the evening peak. Each night the operator decides how much to top it up from hydro. A clear day needs no top-up; a cloudy day needs a lot. The cost counts the evening peak hydro must supply (weight 2) plus the overnight top-up energy (weight 1).

**Read the caveats first.** The load profile is synthetic, hourly solar is a clear-sky bell scaled to the real daily total, and sizes are assumptions in `BatteryConfig`. Daily solar and its forecast skill are real.

In [ ]:
from tana.config import BatteryConfig
from tana.pipeline import load_daily, analyse_battery
from tana import battery as bat

bc = BatteryConfig()
try:
    daily = load_daily(cfg, bc, synthetic=SYNTHETIC)
    BT = analyse_battery(cfg, bc, daily)
    display(Markdown("**Day-ahead solar forecast error (test years)**"))
    display(BT["fscores"].round(3))
    display(Markdown("**Policy comparison**"))
    display(BT["table"].round(1))
except Exception as exc:
    BT = None
    print("Battery section skipped:", exc)

In [ ]:
if BT is not None:
    dc = bat.day_detail("ar1", BT["clear_day"], BT["fc"], bc)
    dk = bat.day_detail("ar1", BT["cloudy_day"], BT["fc"], bc)
    show(plotting.fig_battery_days(dc, dk, bc, (f"Clear day {BT['clear_day']:%d %b %Y}", f"Cloudy day {BT['cloudy_day']:%d %b %Y}")))
    show(plotting.fig_battery_scorecard(BT["table"], bc))
    show(plotting.fig_battery_months(BT["runs"]))
    show(plotting.fig_battery_sensitivity(BT["sens"]))

**How to read it.** Compare each forecast-aware policy with the *best static* policy, not with the worst one. Perfect foresight shows the ceiling. The monthly chart shows whether the forecast matters mainly in the cloudy rainy season, and the sensitivity charts show when it does and does not pay. If forecast-aware rules barely beat a static rule, that is the finding.

## Limitations (state these when you present)

- One 0.5-degree grid cell; NASA POWER rainfall is a satellite and reanalysis blend, not a gauge record.
- Inflow is a simple lagged-rainfall rule, not a hydrological model. Reservoir size, demand and plant efficiency are assumptions.
- Test period is about 10 years; the differences between rules are described, not statistically tested.
- No grid data, transmission limits or electricity prices were used.
- Monthly wind means understate variability, so wind energy is approximated through a Weibull assumption.
- The battery experiment uses a synthetic load and a modelled hourly solar shape; it tests decision logic, not plant performance.